## Configuration search on own NN implementation
This code is extremely slow, runtime is about $12$ hours. The results from the last run are avaliable in the `.json`-file

In [ ]:
from util import *
import autograd.numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os
import time
import json

# Load MNIST data
mnist = fetch_openml('mnist_784', version=1, parser='auto')
X = mnist.data.to_numpy() / 255.0
y = mnist.target.astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=10000 / 70000, random_state=42)
# One-hot encode
train_targets = one_hot_encode(y_train, num_classes=10)
test_targets = one_hot_encode(y_test, num_classes=10)
# Hyperparameter grids
hidden_configs = [
    [30],
    [100],
    [200],
    [100, 50],
    [200, 200]
]
activation_map = {
'leaky_ReLU': {'func': leaky_ReLU, 'der': leaky_ReLU_der},
'ReLU': {'func': ReLU, 'der': ReLU_der},
'sigmoid': {'func': sigmoid, 'der': sigmoid_der},
}
opt_configs = [
    {'name': 'ADAM', 'lrs': [0.0005, 0.001, 0.005], 'rho1': 0.9, 'rho2': 0.999},
    {'name': 'SGD', 'lrs': [0.01, 0.05, 0.1]},
    {'name': 'RMSProp', 'lrs': [0.0005, 0.001, 0.005], 'rho': 0.99}
]
# Fixed
epochs = 15
minibatch_size = 100
input_size = 784
# Log file
log_file = 'mnist_hyperparam_search_results.json'
results = []
if os.path.exists(log_file):
    with open(log_file, 'r') as f:
        results = json.load(f)

# Function to check if combo is already completed
def is_completed(results, hidden, act, opt, lr):
    for res in results:
        if (res.get('hidden_layers') == hidden and
            res.get('activation') == act and
            res.get('optimizer') == opt and
            res.get('lr') == lr and
            res.get('status') == 'success'):
            return True
    return False

# Main search
total_combos = len(hidden_configs) * len(activation_map) * sum(len(conf['lrs']) for conf in opt_configs)
print(f"Starting search: {total_combos} combos (~{total_combos * 5 / 60:.1f} hours)")
combo_idx = 0
for hidden_layers in hidden_configs:
    for act_name, act_info in activation_map.items():
        layer_output_sizes = hidden_layers + [10]
        activation_funcs = [act_info['func']] * len(hidden_layers) + [linear]
        activation_ders = [act_info['der']] * len(hidden_layers) + [linear_der]
        for opt_conf in opt_configs:
            opt_name = opt_conf['name']
            for learning_rate in opt_conf['lrs']:
                combo_idx += 1
                print(f"\nCombo {combo_idx}/{total_combos}: hidden={hidden_layers}, act={act_name}, opt={opt_name}, lr={learning_rate}")
                
                if is_completed(results, hidden_layers, act_name, opt_name, learning_rate):
                    print("Skipping completed combo")
                    continue
                
                try:
                    start_time = time.perf_counter()
                    nn = NeuralNetwork(
                        network_input_size=input_size,
                        layer_output_sizes=layer_output_sizes,
                        activation_funcs=activation_funcs,
                        activation_ders=activation_ders,
                        cost_fun=cross_entropy_logits,
                        cost_der=cross_entropy_logits_der,
                        lam=0.0,
                        regularizer=None
                    )
                    if opt_name == 'ADAM':
                        nn.ADAM_stochastic(X_train, train_targets, learning_rate, opt_conf['rho1'], opt_conf['rho2'], epochs, minibatch_size)
                    elif opt_name == 'RMSProp':
                        nn.RMSProp_stochastic(X_train, train_targets, learning_rate, opt_conf['rho'], epochs, minibatch_size)
                    elif opt_name == 'SGD':
                        nn.gradient_descent_stochastic(X_train, train_targets, learning_rate, epochs, minibatch_size)
                    final_cost = nn.cost(nn.predict_batch(X_train), train_targets)
                    train_preds = nn.predict_labels(X_train)
                    train_acc = accuracy_score(y_train, train_preds)
                    test_preds = nn.predict_labels(X_test)
                    test_acc = accuracy_score(y_test, test_preds)
                    elapsed = (time.perf_counter() - start_time) / 60.0
                    result = {
                        'hidden_layers': hidden_layers,
                        'activation': act_name,
                        'optimizer': opt_name,
                        'lr': learning_rate,
                        'epochs': epochs,
                        'minibatch_size': minibatch_size,
                        'train_acc': train_acc,
                        'test_acc': test_acc,
                        'final_train_cost': final_cost,
                        'time_min': elapsed,
                        'status': 'success'
                    }
                except Exception as e:
                    print(f"Error in combo {combo_idx}: {str(e)}")
                    result = {
                        'hidden_layers': hidden_layers,
                        'activation': act_name,
                        'optimizer': opt_name,
                        'lr': learning_rate,
                        'epochs': epochs,
                        'minibatch_size': minibatch_size,
                        'train_acc': None,
                        'test_acc': None,
                        'final_train_cost': None,
                        'time_min': None,
                        'status': 'failed',
                        'error': str(e)
                    }
                    elapsed = (time.perf_counter() - start_time) / 60.0 if 'start_time' in locals() else 0
                results.append(result)
                print(f"Results: {result['status']}, Time={elapsed:.2f} min")
                with open(log_file, 'w') as f:
                    json.dump(results, f, indent=4)
print("\nSearch complete. Results in", log_file)

## Configuration search with PyTorch
Each configuration takes $\sim 1$ minute to train, total runtime is about $2$ hours

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score
import os
import time
import json
import numpy as np

# Load MNIST data
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=100, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=False)

# Hyperparameter grids (matching the custom code)
hidden_configs = [
    [30],
    [100],
    [200],
    [100, 50],
    [200, 200]
]

activation_map = {
    'leaky_ReLU': nn.LeakyReLU(negative_slope=0.01),
    'ReLU': nn.ReLU(),
    'sigmoid': nn.Sigmoid()
}

opt_configs = [
    {'name': 'ADAM', 'lrs': [0.0005, 0.001, 0.005]},
    {'name': 'SGD', 'lrs': [0.01, 0.05, 0.1]},
    {'name': 'RMSProp', 'lrs': [0.0005, 0.001, 0.005]}
]

# Fixed params
epochs = 15
minibatch_size = 100  # Already set in loader
input_size = 784
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Log file
log_file = 'mnist_pytorch_search_results.json'
results = []
if os.path.exists(log_file):
    with open(log_file, 'r') as f:
        results = json.load(f)

# Function to check if combo is already completed
def is_completed(results, hidden, act, opt, lr):
    for res in results:
        if (res.get('hidden_layers') == hidden and
            res.get('activation') == act and
            res.get('optimizer') == opt and
            res.get('lr') == lr and
            res.get('status') == 'success'):
            return True
    return False

# Define MLP model
class MLP(nn.Module):
    def __init__(self, input_size, hidden_layers, activation):
        super(MLP, self).__init__()
        layers = []
        in_features = input_size
        for out_features in hidden_layers:
            layers.append(nn.Linear(in_features, out_features))
            layers.append(activation)
            in_features = out_features
        layers.append(nn.Linear(in_features, 10))  # Output layer
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(-1, input_size)
        return self.network(x)

# Training function
def train_model(model, optimizer, criterion, epochs, train_loader, device):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch}: Loss = {total_loss:.2f}")

# Evaluation function
def evaluate_model(model, loader, device):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for data, target in loader:
            data = data.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True).squeeze().cpu().numpy()
            preds.extend(pred)
            labels.extend(target.numpy())
    return accuracy_score(labels, preds)

# Main search
total_combos = len(hidden_configs) * len(activation_map) * sum(len(conf['lrs']) for conf in opt_configs)
print(f"Starting PyTorch search: {total_combos} combos")
combo_idx = 0
for hidden_layers in hidden_configs:
    for act_name, activation in activation_map.items():
        for opt_conf in opt_configs:
            opt_name = opt_conf['name']
            for learning_rate in opt_conf['lrs']:
                combo_idx += 1
                print(f"\nCombo {combo_idx}/{total_combos}: hidden={hidden_layers}, act={act_name}, opt={opt_name}, lr={learning_rate}")
                
                if is_completed(results, hidden_layers, act_name, opt_name, learning_rate):
                    print("Skipping completed combo")
                    continue
                
                try:
                    start_time = time.perf_counter()
                    
                    model = MLP(input_size, hidden_layers, activation).to(device)
                    
                    criterion = nn.CrossEntropyLoss()
                    
                    if opt_name == 'ADAM':
                        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
                    elif opt_name == 'SGD':
                        optimizer = optim.SGD(model.parameters(), lr=learning_rate)
                    elif opt_name == 'RMSProp':
                        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate)
                    
                    train_model(model, optimizer, criterion, epochs, train_loader, device)
                    
                    train_acc = evaluate_model(model, train_loader, device)
                    test_acc = evaluate_model(model, test_loader, device)
                    
                    # Compute final train loss
                    model.eval()
                    final_cost = 0
                    with torch.no_grad():
                        for data, target in train_loader:
                            data, target = data.to(device), target.to(device)
                            output = model(data)
                            final_cost += criterion(output, target).item()
                    
                    elapsed = (time.perf_counter() - start_time) / 60.0
                    
                    result = {
                        'hidden_layers': hidden_layers,
                        'activation': act_name,
                        'optimizer': opt_name,
                        'lr': learning_rate,
                        'epochs': epochs,
                        'minibatch_size': minibatch_size,
                        'train_acc': train_acc,
                        'test_acc': test_acc,
                        'final_train_cost': final_cost,
                        'time_min': elapsed,
                        'status': 'success'
                    }
                except Exception as e:
                    print(f"Error in combo {combo_idx}: {str(e)}")
                    result = {
                        'hidden_layers': hidden_layers,
                        'activation': act_name,
                        'optimizer': opt_name,
                        'lr': learning_rate,
                        'epochs': epochs,
                        'minibatch_size': minibatch_size,
                        'train_acc': None,
                        'test_acc': None,
                        'final_train_cost': None,
                        'time_min': None,
                        'status': 'failed',
                        'error': str(e)
                    }
                    elapsed = (time.perf_counter() - start_time) / 60.0 if 'start_time' in locals() else 0
                
                results.append(result)
                print(f"Results: {result['status']}, Time={elapsed:.2f} min")
                with open(log_file, 'w') as f:
                    json.dump(results, f, indent=4)

print("\nSearch complete. Results in", log_file)

## Plotting results from own NN configuration search
Plot the results from the `.json`-file

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json

# Load the results JSON
with open('mnist_hyperparam_search_results.json', 'r') as f:
    results = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(results)

# Filter successful runs and exclude low-acc outliers (likely diverged training)
df_success = df[(df['status'] == 'success') & (df['test_acc'] > 0.5)]

# Remove rows with NaN in key columns (e.g., test_acc, time_min)
df_success = df_success.dropna(subset=['test_acc', 'time_min', 'lr'])

# Convert hidden_layers to string for categorical plotting
df_success['hidden_str'] = df_success['hidden_layers'].apply(lambda x: '-'.join(map(str, x)))

# Plot 1: Test Accuracy by Activation Function (boxplot)
plt.figure(dpi = 300, figsize=(10, 6))
sns.boxplot(x='activation', y='test_acc', data=df_success)
plt.title('Test Accuracy by Activation Function')
plt.ylabel('Test Accuracy')
plt.xlabel('Activation')
plt.ylim(0.5, 1)
plt.savefig('figs/MNIST/test_acc_by_activation.png')
plt.show()
plt.close()

# Plot 2: Test Accuracy by Optimizer (boxplot)
plt.figure(dpi = 300, figsize=(10, 6))
sns.boxplot(x='optimizer', y='test_acc', data=df_success)
plt.title('Test Accuracy by Optimizer')
plt.ylabel('Test Accuracy')
plt.xlabel('Optimizer')
plt.ylim(0.5, 1)
plt.savefig('figs/MNIST/test_acc_by_optimizer.png')
plt.show()
plt.close()

# Plot 3: Test Accuracy by Hidden Layer Configuration (boxplot)
plt.figure(dpi = 300, figsize=(12, 6))
sns.boxplot(x='hidden_str', y='test_acc', data=df_success)
plt.title('Test Accuracy by Hidden Layer Configuration')
plt.ylabel('Test Accuracy')
plt.xlabel('Hidden Layer Configuration')
plt.ylim(0.5, 1)
plt.savefig('figs/MNIST/test_acc_by_hidden.png')
plt.show()
plt.close()

# Plot 4: Test Accuracy vs Learning Rate by Optimizer (lineplot)
plt.figure(dpi=300, figsize=(10, 6))
sns.lineplot(x='lr', y='test_acc', hue='optimizer', data=df_success, marker='o')
plt.title('Test Accuracy vs Learning Rate by Optimizer')
plt.ylabel('Test Accuracy')
plt.xlabel('Learning Rate, log scale')
plt.xscale('log')  # Since LRs vary by orders
plt.ylim(0.5, 1)
plt.legend(title='Optimizer')
plt.savefig('figs/MNIST/test_acc_vs_lr_by_opt.png')
plt.show()
plt.close()

# Plot 5: Test Accuracy by Activation and Optimizer (heatmap average)
pivot = df_success.pivot_table(values='test_acc', index='activation', columns='optimizer', aggfunc='mean')
plt.figure(dpi=300)
sns.heatmap(pivot, annot=True, vmin=0.8, vmax=0.95)
plt.title('Average Test Accuracy by Activation and Optimizer')
plt.xlabel('Optimizer')
plt.ylabel('Activation function')
plt.savefig('figs/MNIST/test_acc_heatmap_act_opt.png')
plt.show()
plt.close()

# Plot 6: Training Time by Hidden Layer Configuration (boxplot)
plt.figure(dpi = 300, figsize=(12, 6))
sns.boxplot(x='hidden_str', y='time_min', data=df_success)
plt.title('Training Time by Hidden Layer Configuration')
plt.ylabel('Time (minutes)')
plt.xlabel('Hidden Layer Configuration')
plt.savefig('figs/MNIST/time_by_hidden.png')
plt.show()
plt.close()

# New Plot 7: Test Accuracy vs Learning Rate by Activation (lineplot)
plt.figure(dpi = 300, figsize=(10, 6))
sns.lineplot(x='lr', y='test_acc', hue='activation', data=df_success, marker='o')
plt.title('Test Accuracy vs Learning Rate by Activation')
plt.ylabel('Test Accuracy')
plt.xlabel('Learning Rate, log scale')
plt.xscale('log')
plt.legend(title='Activation')
plt.ylim(0.5, 1)
plt.savefig('figs/MNIST/test_acc_vs_lr_by_act.png')
plt.show()
plt.close()

## Plotting results from PyTorch configuration search
Plotting results from the `.json`-file

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
# Set larger font scale for all plots


# Load the results JSON
with open('mnist_pytorch_search_results.json', 'r') as f:
    results = json.load(f)
# Convert to DataFrame
df = pd.DataFrame(results)
# Filter successful runs
df_success = df[df['status'] == 'success']
# Convert hidden_layers to string for categorical plotting
df_success['hidden_str'] = df_success['hidden_layers'].apply(lambda x: '-'.join(map(str, x)))
# Plot 1: Test Accuracy by Activation Function (boxplot)
plt.figure(figsize=(10, 6))
sns.boxplot(x='activation', y='test_acc', data=df_success)
plt.title('Test Accuracy by Activation Function')
plt.ylabel('Test Accuracy')
plt.xlabel('Activation')
plt.ylim(0.85, 1)
plt.savefig('figs/MNIST/ptest_acc_by_activation.png')
plt.show()
plt.close()

# Plot 2: Test Accuracy by Optimizer (boxplot)
plt.figure(figsize=(10, 6))
sns.boxplot(x='optimizer', y='test_acc', data=df_success)
plt.title('Test Accuracy by Optimizer')
plt.ylabel('Test Accuracy')
plt.xlabel('Optimizer')
plt.ylim(0.85, 1)
plt.savefig('figs/MNIST/ptest_acc_by_optimizer.png')
plt.show()
plt.close()

# Plot 3: Test Accuracy by Hidden Layer Configuration (barplot with error bars)
plt.figure(figsize=(12, 6))
sns.boxplot(x='hidden_str', y='test_acc', data=df_success)
plt.title('Test Accuracy by Hidden Layer Configuration')
plt.ylabel('Test Accuracy')
plt.xlabel('Hidden Layers')
plt.ylim(0.85, 1)
plt.savefig('figs/MNIST/ptest_acc_by_hidden.png')
plt.show()
plt.close()

# Plot 4: Test Accuracy vs Learning Rate by Optimizer (lineplot)
plt.figure(figsize=(12, 6))
sns.lineplot(x='lr', y='test_acc', hue='optimizer', data=df_success, marker='o')
plt.title('Test Accuracy vs Learning Rate by Optimizer')
plt.ylabel('Test Accuracy')
plt.xlabel('Learning Rate')
plt.xscale('log') # Since LRs vary by orders
plt.ylim(0.9, 1)
plt.savefig('figs/MNIST/ptest_acc_vs_lr_by_opt.png')
plt.show()
plt.close()

# Plot 5: Test Accuracy by Activation and Optimizer (heatmap average)
#include 3 decimal places in this plot
pivot = df_success.pivot_table(values='test_acc', index='activation', columns='optimizer', aggfunc='mean')
plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=".3f", vmin=0.93, vmax=0.98)
plt.title('Average Test Accuracy by Activation and Optimizer')
plt.savefig('figs/MNIST/ptest_acc_heatmap_act_opt.png')
plt.show()
plt.close()

# Plot 6: Training Time by Hidden Layer Configuration (boxplot)
plt.figure(figsize=(12, 6))
sns.boxplot(x='hidden_str', y='time_min', data=df_success)
plt.title('Training Time by Hidden Layer Configuration')
plt.ylabel('Time (minutes)')
plt.xlabel('Hidden Layers')
plt.savefig('figs/MNIST/ptime_by_hidden.png')
plt.show()
plt.close()